In [ ]:
!pip install -q transformers datasets accelerate

In [ ]:
from google.colab import files          # Colab file upload helper
uploaded = files.upload()

Saving email.csv to email.csv


In [ ]:
import re, torch, numpy as np, pandas as pd     # preprocessing                           # utils + PyTorch
from datasets import Dataset          # used for compatibility                                     # HF dataset
from sklearn.model_selection import train_test_split       # data splittiung               # data split
from sklearn.metrics import accuracy_score, classification_report        # classification evaluation
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)     # BERT + Trainer


In [ ]:
df = pd.read_csv("email.csv")
df = df[["Category", "Message"]]
print("Shape:", df.shape)                 # rows, columns
df.head()

Shape: (5573, 2)


,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
def clean_text(Message):
    text = str(Message).lower()                       # convert to string and lowercase
    text = re.sub(r"[^a-z\s.,!?']", " ", Message)     # remove anything except letters and basic punctuation
  #  text = re.sub(r"\s+", " ", Message)               # collapse multiple spaces into one
    return text.strip()                            # remove spaces at the start and end

df["Message"] = df["Message"].apply(clean_text)          # clean every row
df.head()

,Category,Message
0,ham,"o until jurong point, crazy.. vailable only i..."
1,ham,k lar... oking wif u oni...
2,spam,ree entry in a wkly comp to win up final...
3,ham,dun say so early hor... c already then say...
4,ham,"ah don't think he goes to usf, he lives arou..."


In [ ]:
# the two maps, written out plainly
label2id = {"ham": 0, "spam": 1}   # word  -> number
id2label = {0: "ham", 1: "spam"}   # number -> word

# turn the sentiment words into numbers
df["label"] = df["Category"].map(label2id)               # words -> 0/1/2

# Drop rows where 'label' is NaN due to invalid category entries
df.dropna(subset=["label"], inplace=True)

# Convert label column to integer type to ensure no float NaNs persist
df["label"] = df["label"].astype(int)

#print(df.head())
print(label2id)
print(id2label)
print(df["Category"].value_counts())
print(df["label"].value_counts())

{'ham': 0, 'spam': 1}
{0: 'ham', 1: 'spam'}
Category
ham     4825
spam     747
Name: count, dtype: int64
label
0    4825
1     747
Name: count, dtype: int64


In [ ]:
print(f"NaNs in df['label'] before split: {df['label'].isnull().sum()}")
print(f"dtype of df['label'] before split: {df['label'].dtype}")
train_df, test_df = train_test_split(         # split into train and test
    df[["Message", "label"]],                    # use ALL rows; only need text + label
    test_size=0.2,                            # 20% goes to test
    random_state=42,                          # reproducible split
    stratify=df["label"],                     # keep class ratio in both sets
)
print("Train:", len(train_df), "Test:", len(test_df))
print(train_df["label"].value_counts())
print(test_df.iloc[0,0])

NaNs in df['label'] before split: 0
dtype of df['label'] before split: int64
Train: 4457 Test: 1115
label
0    3859
1     598
Name: count, dtype: int64
o need to buy lunch for me..   eat maggi mee..


In [ ]:
model_name = "distilbert-base-uncased"                       # which BERT to use
tokenizer = AutoTokenizer.from_pretrained(model_name)  # load its tokenizer

def tokenize(batch):                                   # function applied to data
    return tokenizer(batch["Message"],                    # the text column
                     padding="max_length",             # pad short tweets
                     truncation=True,                  # cut long tweets
                     max_length=128)                   # fixed length = 128 tokens

train_ds = Dataset.from_pandas(train_df).map(tokenize, batched=True)  # tokenize train
test_ds  = Dataset.from_pandas(test_df).map(tokenize, batched=True)  # tokenize test

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/4457 [00:00<?, ? examples/s]

Map:   0%|          | 0/1115 [00:00<?, ? examples/s]

In [ ]:
train_ds

Dataset({
    features: ['Message', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4457
})

In [ ]:
print("first message:\n",train_ds["Message"][0])
print(train_ds["input_ids"][0])
print(len(train_ds["input_ids"][0])) # it is also known as token id
print(train_ds["attention_mask"][0])

first message:
 e will, you guys close?
[101, 1041, 2097, 1010, 2017, 4364, 2485, 1029, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
128
[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(  # BERT + classifier on top
    model_name,            # same bert-base-uncased
    num_labels=3,          # 3 classes: neg / neu / pos
    id2label=id2label,     # readable output labels
    label2id=label2id,     # reverse map
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def compute_metrics(eval_pred):                       # Trainer calls this automatically during evaluation
    logits, labels = eval_pred                        # unpack: logits = raw model scores, labels = true answers
    preds = np.argmax(logits, axis=-1)                # pick the class with the highest score = prediction
    accuracy = accuracy_score(labels, preds)          # compare predictions vs true labels to get accuracy
    return {"accuracy": accuracy}                      # return as a dict so Trainer can log it by name

In [ ]:
args = TrainingArguments(
    output_dir="results",                 # where checkpoints & logs are saved (required)
    eval_strategy="epoch",                # evaluate after every epoch
    save_strategy="epoch",                # save a checkpoint after every epoch
    num_train_epochs=3,                   # full passes through the training data
    per_device_train_batch_size=8,        # training samples per batch
    per_device_eval_batch_size=8,         # evaluation samples per batch
    learning_rate=2e-5,                   # small step size for fine-tuning
    weight_decay=0.01,                    # regularization to reduce overfitting
    logging_steps=50,                     # log metrics every 50 steps
    load_best_model_at_end=True,          # reload the best checkpoint after training
    metric_for_best_model="accuracy",     # judge "best" by accuracy
)

In [ ]:
trainer = Trainer(                         # bundles everything together
    model=model,                           # the BERT model
    args=args,                             # the settings above
    train_dataset=train_ds,                # data to learn from
    eval_dataset=test_ds,                  # data to check on
    compute_metrics=compute_metrics,       # how to score
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.105730,0.096101,0.978475
2,0.020065,0.094122,0.981166
3,0.015544,0.094248,0.982960


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1674, training_loss=0.06535482975423977, metrics={'train_runtime': 179.6933, 'train_samples_per_second': 74.41, 'train_steps_per_second': 9.316, 'total_flos': 442805396857344.0, 'train_loss': 0.06535482975423977, 'epoch': 3.0})

In [ ]:
preds = trainer.predict(test_ds)                 # predict on the test set
y_pred = np.argmax(preds.predictions, axis=-1)   # chosen class per tweet
y_true = preds.label_ids                         # the true classes

print(classification_report(                     # precision/recall/F1 per class
    y_true, y_pred, target_names=["ham","spam"]
))

              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       966
        spam       0.99      0.88      0.93       149

    accuracy                           0.98      1115
   macro avg       0.99      0.94      0.96      1115
weighted avg       0.98      0.98      0.98      1115



In [ ]:
from transformers import pipeline                 # easy inference wrapper

clf = pipeline("text-classification",             # task
               model=model, tokenizer=tokenizer)  # use our fine-tuned model

print(clf(clean_text("o until jurong point, crazy.. vailable only i...")))     # clean then predict
print(clf(clean_text("Free entry in 2 a wkly comp to win FA Cup fina...")))  # negative example

[{'label': 'ham', 'score': 0.9995226860046387}]
[{'label': 'spam', 'score': 0.9989159107208252}]


In [ ]:
trainer.save_model("distil bert-email-sentiment_model")          # save model weights
tokenizer.save_pretrained("distil bert-email-sentiment_model")   # save tokenizer


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('distil bert-email-sentiment_model/tokenizer_config.json',
 'distil bert-email-sentiment_model/tokenizer.json')